In [ ]:
event = {
    "id": "https://example.com/event1",
    "label": "tweede expeditie naar nias",
    "title": "tweede expeditie naar nias",
    "timeSpan_start": "1863-01-01",
    "timeSpan_end": "1863-01-01",
    "date_y": 1863,
    "date_m": 1,
    "date_d": 1,
    "UTC_DATE": "1863-01-01T00:00:00Z",
    "fulltext": "",
    "entities": []
}

Given an event extract the articles

Remark: semantic search for the events are whole different issue

In [ ]:
import xmltodict as xd
import json
import os
import requests
from utils import extract_str

def parse_resp_events(response, event):
    data = xd.parse(response.content, xml_attribs=False)
    raw_records = data['srw:searchRetrieveResponse']['srw:records']['srw:record']
    if not isinstance(raw_records, list):
        raw_records = [raw_records]
   
    # print(f"raw_records: {len(raw_records)} for response: {response.url}")

    # THIS IS TEMPORARY
    dir = os.getcwd()
    
    for record in raw_records:
        record_data = record['srw:recordData']
        # Load zones as json
        record_data['zones'] = json.loads(record_data['zones'])
        # Get OCR (text content of response)
        ocr_url = record_data['dc:identifier']
        with requests.get(ocr_url) as ocr_response:
            record_data['ocr'] = ocr_response.text
        record_data['event_id'] = event.get('id', '')
        record_data['event_title'] = event.get('title', '')

        identifier = "_".join(extract_str(record_data['dc:identifier'], '?urn=').split(':')[:-1])  # Extract identifier and remove 'ddd:' prefix
        filename = os.path.join(dir, 'DST', identifier + '.json')

        # print(f"{type(record_data)} for record_data type, {filename} for filename")

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(record_data, f, indent=2, ensure_ascii=False)

        

In [ ]:
base_EVENT_BASED_URL = f"https://jsru.kb.nl/sru/sru?version=1.2&operation=searchRetrieve&x-collection=DDD_artikel&recordSchema=indexing&startRecord=1&maximumRecords=1&query=(content all \"%s\") AND (date within \"%s %s\")"
# iter_EVENT_BASED_URL = f"https://jsru.kb.nl/sru/sru?version=1.2&operation=searchRetrieve&x-collection=DDD_artikel&recordSchema=indexing&startRecord=%s&maximumRecords=%s&query=\"%s\" AND (date within \"%s %s\")"
iter_EVENT_BASED_URL = f"https://jsru.kb.nl/sru/sru?version=1.2&operation=searchRetrieve&x-collection=DDD_artikel&startRecord=%s&maximumRecords=%s&query=(content all \"%s\") AND (date within \"%s %s\")&recordSchema=ddd&x-fields=zones"

import pandas as pd

import requests
import lxml.etree
def get_article_by_event(event:json):
    title = event.get("title", "")
    date_y = event.get("date_y", "")
    # print(f"Title: {title}, Fulltext: {fulltext}, Date: {date_y}")

    base_url = base_EVENT_BASED_URL % (title, date_y, date_y+10)
    print(f"Fetching articles for event: {title} with base URL: {base_url}")

    resp = requests.get(base_url)
    data = lxml.etree.fromstring(resp.content)
    total_nr_results = 0
    for i in data.iter():
        if i.tag == '{http://www.loc.gov/zing/srw/}numberOfRecords':
            total_nr_results = int(i.text)
            break
    print(f"Total results: {total_nr_results} for event: {title}")

    if total_nr_results == 0:
        print(f"No results found for event: {title}")
        return
    
    inv = 10 if total_nr_results > 10 else total_nr_results

    result_dicts = []
    for start in range(1, total_nr_results+1, inv):
        paged_url = iter_EVENT_BASED_URL % (start, inv, title, date_y, date_y+10)
        # print(f"Fetching articles for event: {title} with paged URL: {paged_url}")
        paged_resp = requests.get(paged_url)
        parse_resp_events(paged_resp, event)

    # # convert list of dicts to dataframe
    # df = pd.DataFrame(result_dicts)
    # df.to_csv(f"{title.replace(' ', '_')}_articles.csv", index=False)
    

In [ ]:
get_article_by_event(event)

### Apply NER on each article

In [1]:
# We need the latest version of GLiNER, which is for some reason not available 
# through pypy (https://github.com/urchade/GLiNER/issues/360), we can install
# it from source.
import os
os.chdir('../GLiNER')

# !git clone https://github.com/urchade/GLiNER
# os.chdir('GLiNER')
!pip install -r requirements.txt -q
!pip install . -q


os.chdir('../src')
os.getcwd()

'/Users/sarah_shoilee/codeProjects/linking_delpher/src'

In [2]:
import torch
import transformers

print(torch.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())

2.5.1
5.6.2
False


In [3]:
import gliner
assert gliner.__version__ == '0.2.27'
print(gliner.__version__)

0.2.27


In [4]:
from pathlib import Path

from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import nltk

# from data_utils import prepare_data, convert_to_dataset, ner_tags_to_spans

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sarah_shoilee/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
# Rename tags in datasets to have full tag names, which are used in the model

# English classes
ner_tag_to_class_mapping = {
    'O': 'O',
    'B-PER': 'B-person',
    'I-PER': 'I-person',
    'B-ORG': 'B-organization',
    'I-ORG': 'I-organization',
    'B-LOC': 'B-location',
    'I-LOC': 'I-location',
    'B-EVT': 'B-historical_event',
    'I-EVT': 'I-historical_event'
}

# Dutch classes
ner_tag_to_class_mapping = {
    'O': 'O',
    'B-PER': 'B-persoon',
    'I-PER': 'I-persoon',
    'B-ORG': 'B-organisatie',
    'I-ORG': 'I-organisatie',
    'B-LOC': 'B-locatie',
    'I-LOC': 'I-locatie',
    'B-EVT': 'B-historisch_event',
    'I-EVT': 'I-historisch_event'
}

# Generate label list
label_list = list(ner_tag_to_class_mapping.values())

# label_map = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}
tag_to_id = {label: i for i, label in enumerate(label_list)}
tag_to_id

{'O': 0,
 'B-persoon': 1,
 'I-persoon': 2,
 'B-organisatie': 3,
 'I-organisatie': 4,
 'B-locatie': 5,
 'I-locatie': 6,
 'B-historisch_event': 7,
 'I-historisch_event': 8}

In [6]:
from gliner import GLiNER
import gliner
import torch

In [7]:
model_id = "urchade/gliner_small"
model_id = "EmergentMethods/gliner_large_news-v2.1"
model_id = "urchade/gliner_multi-v2.1"


if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"


model = GLiNER.from_pretrained(
    model_id, 
    load_tokenizer=True, 
    map_location=device
)

/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [8]:
# use_fine_grained = True
use_fine_grained = False

if not use_fine_grained:
    entity_types = list(set([i.split("-")[-1] for i in label_list if i != "O"]))
    # list(set([i.split("-")[-1] for i in ner_tag_to_class_mapping.values() if i != "O"]))
else:
    pass
    # entity_types = list(set(fine_grained_labels.keys()))

print(f"{len(entity_types)} entity types: {entity_types}")

4 entity types: ['persoon', 'historisch_event', 'locatie', 'organisatie']


In [9]:
import re
from nltk.tokenize import sent_tokenize

def chunk_sentence(text: str, max_token: int, tokenizer=None):
    """Split a text into chunks where each chunk has at most max_token tokens."""
    if max_token <= 0:
        raise ValueError("max_token must be > 0")

    if tokenizer is None:
        tokenizer = lambda s: re.findall(r"\S+", s)

    chunks = []
    for sentence in sent_tokenize(text):
        tokens = tokenizer(sentence)
        if len(tokens) <= max_token:
            chunks.append(sentence.strip())
            continue

        current = []
        for token in tokens:
            if len(current) >= max_token:
                chunks.append(" ".join(current))
                current = [token]
            else:
                current.append(token)

        if current:
            chunks.append(" ".join(current))

    return chunks


In [11]:

def extract_names_entities(text:str):
    entities = model.predict_entities(text, entity_types)

    person_entities = [e for e in entities if e["label"] == "persoon"]
    location_entities = [e for e in entities if e["label"] == "locatie"]
    organization_entities = [e for e in entities if e["label"] == "organisatie"]
    historical_event_entities = [e for e in entities if e["label"] == "historisch_event"]

    return {
        "person": person_entities,
        "geographic location": location_entities,
        "organization": organization_entities,
        "historical_event": historical_event_entities
    }

In [12]:
import lxml.etree
import json
import os
from tqdm import tqdm

# create a list of extracted entity records
extracted_records = []

# traverse through all files in a directory
DIR = os.path.join(os.getcwd(), 'DST')
output_path = os.path.join(DIR, 'extracted_entities.csv')

for filename in tqdm(os.listdir(DIR), desc="Processing JSON files"):
    if not filename.endswith('.json'):
        continue

    filepath = os.path.join(DIR, filename)

    # read the json file and extract the OCR content
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    ocr_content = data.get('ocr', '')
    # read OCR content as XML and separate text by paragraphs
    ocr_content = lxml.etree.fromstring(ocr_content.encode('utf-8')).xpath('//p/text()')

    data['person'] = []
    data['geographic location'] = []
    data['organization'] = []
    data['historical_event'] = []

    # print(f"Extracting entities for file: {filename} with OCR content length: {len(ocr_content)}")
    rows = []
    try:
        for i, paragraph in enumerate(ocr_content):
            for j, sentence in enumerate(chunk_sentence(paragraph, max_token=200)):
                entities = extract_names_entities(sentence)
                # print(f"Extracted {len(entities['person'])} persons, {len(entities['geographic location'])} locations, {len(entities['organization'])} orgs in sentence {j}")
                data['person'] += entities['person']
                data['geographic location'] += entities['geographic location']
                data['organization'] += entities['organization']
                data['historical_event'] += entities['historical_event']
                rows.append({
                    'filename': filename,
                    'paragraph': paragraph,
                    'sentence': sentence,
                    'person': entities['person'],
                    'geographic location': entities['geographic location'],
                    'organization': entities['organization'],
                    'historical_event': entities['historical_event']
                })
    except KeyboardInterrupt:
        print('KeyboardInterrupt caught — saving partial results...')
    finally:
        if rows:
            file_exists = os.path.exists(output_path)
            pd.DataFrame(rows).to_csv(
                output_path,
                mode='a+' if file_exists else 'w',
                header=not file_exists,
                index=False
            )
            # print(f'Saved extracted entities to {output_path}')

    # print('==========================================================')
    # print(f"Extracted entities for file: {filename} - \n\tpersons: {data['person']} , \n\tlocations: {data['geographic location']} , \n\torganizations: {data['organization']} , \n\thistorical_events: {data['historical_event']}")

    # add the extracted entities to the json file and save it again
    data['entities'] = extract_names_entities(ocr_content)  # This is a placeholder for the actual entity extraction function
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


Processing JSON files: 100%|██████████| 206/206 [3:02:54<00:00, 53.28s/it]   


Disambiguate person within article

Disambiguate person within event (across article)

Make list of person dictionary 